# 02 · Preparación de Datos

**Objetivo:** Construir un pipeline de preprocesado reutilizable para imágenes y datos tabulares.

**Datos de entrada:** `../data/raw/HAM10000_metadata.csv`, `../data/raw/hnmist_28_28_RGB.csv`

**Resultado esperado:** Conjuntos `X_img_train/test`, `X_tab_train/test`, `y_train/test` listos para entrenar cualquiera de los modelos.

> **Nota:** Las celdas de instalación solo es necesario ejecutarlas una vez.

## Instalación de dependencias

In [1]:
#El primer punto es instalar las librerías necesarias:
!pip install pandas 
!pip install scikit-learn
!pip install tensorflow
!pip install matplotlib
!pip install seaborn
!pip install keras
!pip install numpy
!pip install opencv-python
!pip install torch
!pip install torchvision


In [2]:
# Dado que tengo una GPU NVIDIA, instalo PyTorch con soporte CUDA para hacer pruebas con esta GPU:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [3]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.5.1+cu121
12.1
True
NVIDIA GeForce RTX 4070 SUPER


## Descarga del dataset

Descarga los datos desde Google Drive. Si ya tienes los CSVs en `data/raw/`, salta esta celda.

In [4]:
!pip install gdown 


   -------------------------------- ------- 4/5 [gdown]
   ---------------------------------------- 5/5 [gdown]



In [5]:
!gdown --folder https://drive.google.com/drive/folders/1qTxGaojM2eJn0SpHJl2Ujk4zvDkkadPR -O data

Processing file 1n7UewGksx3UZjn34-Q1QJtLXMMp42O_0 HAM10000_metadata.csv
Processing file 1sBEsgtYVo4X7APfg-_miZvpMttfS3eJ4 hnmist_28_28_RGB.csv


Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1n7UewGksx3UZjn34-Q1QJtLXMMp42O_0
To: c:\Users\rammu\Documents\projects\Deeplearning\practica\KC-Deeplearning-\notebooks\data\HAM10000_metadata.csv

  0%|          | 0.00/563k [00:00<?, ?B/s]
 93%|█████████▎| 524k/563k [00:00<00:00, 4.68MB/s]
100%|██████████| 563k/563k [00:00<00:00, 4.63MB/s]
Downloading...
From: https://drive.google.com/uc?id=1sBEsgtYVo4X7APfg-_miZvpMttfS3eJ4
To: c:\Users\rammu\Documents\projects\Deeplearning\practica\KC-Deeplearning-\notebooks\data\hnmist_28_28_RGB.csv

  0%|          | 0.00/91.8M [00:00<?, ?B/s]
  1%|          | 524k/91.8M [00:00<00:25, 3.57MB/s]
  2%|▏         | 1.57M/91.8M [00:00<00:19, 4.75MB/s]
  3%|▎         | 2.62M/91.8M [00:00<00:17, 5.10MB/s]
  4%|▍         | 3.67M/91.8M [00:00<00:16, 5.21MB/s]
  5%|▌         | 4.72M/91.8M [00:00<00:16, 5.36MB/s]
  6%|▋        

## Pipeline de preprocesado centralizado (`utils.py`)

Todo el preprocesado del proyecto vive en `utils.py`, en la raíz del repositorio.
Desde aquí podemos llamar a sus dos funciones principales:

| Función | Uso | Devuelve |
|---|---|---|
| `get_tabular_splits()` | Solo metadatos (notebook 03) | X_tab train/val/test, y train/val/test, LabelEncoder |
| `get_all_splits()` | Imágenes + metadatos (notebooks 04–06) | X_img + X_tab + y train/val/test, LabelEncoder |

**Decisiones de diseño aplicadas en `utils.py`:**
- **Split 70/15/15 con stratify ANTES de transformar**: garantiza que el imputer y el scaler solo ven el train, evitando *data leakage*.
- **Alineamiento por `image_id`** (merge), no por posición: si los CSVs no están en el mismo orden, el emparejamiento posicional asignaría la imagen equivocada a cada diagnóstico.
- **Imputación con mediana del train**: más robusta que la media ante outliers en datos médicos.
- **Normalización de imágenes ÷255**: lleva los píxeles al rango [0, 1].
- **StandardScaler fit solo en train**: si normalizásemos con estadísticas del test, el modelo "vería" información del futuro antes de ser evaluado.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from utils import get_tabular_splits, get_all_splits

# ── Pipeline tabular (notebook 03) ────────────────────────────────────────────
# Metadatos clínicos: edad, sexo, localización → split → imputer → StandardScaler
print("=== Pipeline tabular ===")
X_tab_train, X_tab_val, X_tab_test, y_train, y_val, y_test, le = get_tabular_splits()

# ── Pipeline multimodal (notebooks 04, 05, 06) ────────────────────────────────
# Carga y alinea imágenes + metadatos → split → imputación tabular → normalización imágenes /255
print("\n=== Pipeline multimodal (imágenes + tabular) ===")
(X_img_train, X_img_val, X_img_test,
 X_tab_train, X_tab_val, X_tab_test,
 y_train, y_val, y_test, le) = get_all_splits()

print(f"\nShape imágenes train: {X_img_train.shape}")
print(f"Shape tabular train:  {X_tab_train.shape}")
print(f"Shape etiquetas train: {y_train.shape}")